<a href="https://colab.research.google.com/github/sanjivinicarmel/AgriDataAnalysis2016/blob/main/GRU_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Objective

The objective of this experiment is to build a Deep Learning model using a Gated Recurrent Unit (GRU) network to classify SMS messages as either Spam or Ham (Legitimate).

You will learn how to preprocess text data, convert text into numerical sequences, train a GRU model, and use it to predict whether a new SMS message is spam.

Purpose of the Experiment

Spam messages have become increasingly common in mobile communication. Automatically detecting spam messages helps protect users from fraud, phishing attacks, and unwanted advertisements.

In this experiment, you will:

Understand how text data is preprocessed.
Learn how Tokenization converts text into numbers.
Learn why Padding is required.
Build a GRU-based text classifier.
Train and evaluate the model.
Predict new SMS messages.
Perform hyperparameter tuning and observe the changes in performance.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

In [ ]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sanjivini","key":"ea5ebbcb05f38d759ad4e81ed0269d35"}'}

In [ ]:
import os
import shutil

# Create Kaggle directory
os.makedirs("/root/.kaggle", exist_ok=True)

# Move kaggle.json
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

# Set permissions
os.chmod("/root/.kaggle/kaggle.json", 600)

print("Kaggle API Configured Successfully")

Kaggle API Configured Successfully


In [ ]:
!kaggle datasets download -d uciml/sms-spam-collection-dataset

Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
sms-spam-collection-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
#Extracting
!unzip -o sms-spam-collection-dataset.zip

Archive:  sms-spam-collection-dataset.zip
  inflating: spam.csv                


In [ ]:
#Importing Libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

In [ ]:
#Loading dataset
data = pd.read_csv("spam.csv", encoding="latin-1")

# Select only the required columns
data = data[['v1', 'v2']]

data.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#Exploring dataset
print("Dataset Shape :", data.shape)

print("\nFirst Five Records")
display(data.head())

print("\nClass Distribution")
print(data['v1'].value_counts())

Dataset Shape : (5572, 2)

First Five Records


,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."



Class Distribution
v1
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
#Encoding
#Convert:

#Ham → 0
#Spam → 1
encoder = LabelEncoder()

data['v1'] = encoder.fit_transform(data['v1'])

data.head()

,v1,v2
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
#Separate Features and label
X = data['v2']
y = data['v1']



In [ ]:
#Tokenization
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X)

X = tokenizer.texts_to_sequences(X)

print(X[:5])

[[50, 469, 4410, 841, 751, 657, 64, 8, 1324, 89, 121, 349, 1325, 147, 2987, 1326, 67, 58, 4411, 144], [46, 336, 1495, 470, 6, 1929], [47, 486, 8, 19, 4, 796, 899, 2, 178, 1930, 1199, 658, 1931, 2320, 267, 2321, 71, 1930, 2, 1932, 2, 337, 486, 554, 955, 73, 388, 179, 659, 389, 2988], [6, 245, 152, 23, 379, 2989, 6, 140, 154, 57, 152], [1018, 1, 98, 107, 69, 487, 2, 956, 69, 1933, 218, 111, 471]]


In [ ]:
#Padding
max_length = 100

X = pad_sequences(
    X,
    maxlen=max_length,
    padding="post"
)

print(X.shape)

(5572, 100)


In [ ]:
#Train Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
#Build The GRU Model
model = Sequential()

model.add(
    Embedding(
        input_dim=5000,
        output_dim=64,
        input_length=max_length
    )
)

model.add(
    GRU(64)
)

model.add(
    Dense(
        1,
        activation="sigmoid"
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
#Compile
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Training model
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.20
)

Epoch 1/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 13s 68ms/step - accuracy: 0.8597 - loss: 0.4099 - val_accuracy: 0.8621 - val_loss: 0.4047
Epoch 2/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 10s 67ms/step - accuracy: 0.8670 - loss: 0.3930 - val_accuracy: 0.8621 - val_loss: 0.4017
Epoch 3/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 9s 78ms/step - accuracy: 0.8670 - loss: 0.3945 - val_accuracy: 0.8621 - val_loss: 0.4025
Epoch 4/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 11s 82ms/step - accuracy: 0.8670 - loss: 0.3935 - val_accuracy: 0.8621 - val_loss: 0.4018
Epoch 5/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 9s 67ms/step - accuracy: 0.8670 - loss: 0.3931 - val_accuracy: 0.8621 - val_loss: 0.4033


In [ ]:
#Evaluating Model
loss, accuracy = model.evaluate(X_test, y_test)

print("Loss :", loss)

print("Accuracy :", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.8655 - loss: 0.3969
Loss : 0.39691662788391113
Accuracy : 0.865470826625824


In [ ]:
#Predicting New Message
def predict_message(message):

    sequence = tokenizer.texts_to_sequences([message])

    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post"
    )

    prediction = model.predict(padded)

    if prediction[0][0] > 0.5:
        print("Spam Message")
    else:
        print("Ham Message")

In [ ]:
#Testing the model
predict_message("This is spam message
")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Ham Message
